# Exercise 8: Put all the concepts in Exercise 7 together

Skills:
* Apply all the concepts covered in Exercise 7 for a research question. Know when to use what concept.

References: 
* Exercise 7


### To Do

Narrow down the list of rail routes in CA to 3 groups. Use the SHN network to determine how much of the rail route runs near the SHN. We care only about rail routes that run entirely in CA (use stops to figure this out).

**Near** the interstate, US highway, or state highway is defined by being within a quarter mile. For this exercise, the distinction between interstate, US highway, and state highway is not important; treat any road that shows up in the dataset as "the SHN".

There are theoretically 3 groupings: 
* rail routes that are never within 0.25 miles of the SHN
* rail routes with > 0 but less than half of its length near the SHN 
* rail routes with at least half of its length near the SHN

Provide a table and a chart showing how many rail routes fall into each of the 3 groups by district.

Use a Markdown cell at the end to connect which geospatial concept was applied to which step of the process. The concepts that should be used are `projecting CRS`, `buffering`, `dissolve`, `clipping`, `spatial join`, `overlay`. 

In [ ]:
import geopandas as gpd
import intake
import pandas as pd
from shared_utils import geography_utils
catalog = intake.open_catalog(
    "../_shared_utils/shared_utils/shared_data_catalog.yml")

In [ ]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Import data
districts = catalog.caltrans_districts.read()
highways = catalog.state_highway_network.read()

rail_group = ['0', '1', '2']
routes = catalog.ca_transit_routes.read()
rail_routes = routes[routes.route_type.isin(rail_group)
                    ].reset_index(drop=True)
stops = catalog.ca_transit_stops.read()
rail_stops= stops[stops.route_type.isin(rail_group)
                  ].reset_index(drop=True)

### Look at data

In [ ]:
rail_stops.shape, rail_stops.crs

In [ ]:
rail_stops.sample()

In [ ]:
rail_routes.drop(columns = ['geometry']).sample()

In [ ]:
rail_routes.shape, rail_routes.crs

In [ ]:
highways.drop(columns = ['geometry']).head(6)

In [ ]:
highways.crs

In [ ]:
# highways.sample(5).plot(column = 'Route')

In [ ]:
# highways.sample(5).plot(column = 'Direction')

#### Find only stops in California

In [ ]:
def california():
    # Dissolve district to be California only
    gdf = districts.dissolve()[['geometry','Shape__Area','Shape__Length']]
    return gdf

In [ ]:
ca = california()

In [ ]:
ca.crs == rail_stops.crs

In [ ]:
# Find only stops in California
rail_ca_stops = rail_stops.clip(ca)

In [ ]:
f"{len(rail_stops)-len(rail_ca_stops)} stops are not in CA"

In [ ]:
rail_ca_stops.shape

In [ ]:
rail_ca_stops.agency.unique()

In [ ]:
rail_ca_stops.plot()

In [ ]:
rail_test_keep = ca.clip(rail_stops)

In [ ]:
rail_test_keep.shape

In [ ]:
rail_test_keep.plot()

#### Find only routes in California
* need to dissolve at some point?

In [ ]:
def buffer_geo(gdf):
    gdf = gdf.to_crs("EPSG:2229")
    
    mile = 5_280
    
    gdf = gdf.assign(
    geometry_buffered = gdf.geometry.buffer(0.25*5_280)
    )
    
    gdf = gdf.set_geometry('geometry_buffered')
    
    return gdf

In [ ]:
rail_routes_buffered = buffer_geo(rail_routes)

In [ ]:
# Why is it so much lighter
# Explore takes too long
# rail_routes_buffered.plot()

In [ ]:
rail_routes.crs

In [ ]:
# rail_routes.plot()

In [ ]:
# Find only stops in California by clipping w/o regard to stops
# rail_ca_routes_clipped = rail_routes_buffered.clip(ca.to_crs(rail_routes_buffered.crs))

In [ ]:
# rail_ca_routes_clipped.route_id.nunique()

In [ ]:
# rail_ca_routes_clipped.plot()

In [ ]:
# Merge with routes?
""" rail_ca_routes_m = pd.merge(
    rail_ca_routes_clipped,
    rail_ca_stops[["agency","route_id","route_type"]],
    how = "inner",
    on = ["agency","route_id","route_type"],
    indicator = True
)"""

In [ ]:
# rail_ca_routes_m._merge.value_counts()

In [ ]:
# type(rail_ca_routes_m)

In [ ]:
# rail_ca_routes_m.route_id.nunique()

In [ ]:
# rail_ca_routes_m = rail_ca_routes_m[['agency','route_id','route_type','route_name','geometry']]

In [ ]:
# rail_routes_buffered.crs == rail_ca_stops.crs

In [ ]:
# Why did the sjoin still include stuff 
# Outside of California?
rail_ca_routes_sjoin = gpd.sjoin(
    rail_routes_buffered,
    rail_ca_stops[['geometry']].to_crs(rail_routes_buffered.crs),
    how = "inner",
    predicate = "intersects"
)


In [ ]:
rail_ca_routes_sjoin.shape

In [ ]:
rail_ca_routes_sjoin.route_id.nunique()

In [ ]:
# Check that route IDS are the same for each method
# They are all the same.

#clipped_routes = set(rail_ca_routes_clipped.route_id.unique().tolist())
#merged_routes = set(rail_ca_routes_m.route_id.unique().tolist())
#sjoin_routes = set(rail_ca_routes_sjoin.route_id.unique().tolist())

In [ ]:
#clipped_routes - merged_routes - sjoin_routes

#### Highway intersection

In [ ]:
highways_buffered = buffer_geo(highways)

In [ ]:
# Dissolve because highways have multiple directions
highways_dissolved = highways_buffered.drop(columns = ['geometry','Direction']).dissolve(['Route','County','District', 'RouteType']).reset_index()

In [ ]:
# highways_dissolved.drop(columns = ['geometry_buffered']).sample()

In [ ]:
highways_dissolved.shape, highways_buffered.shape

In [ ]:
highways_dissolved.Route.nunique(), highways_buffered.Route.nunique()

In [ ]:
# highways_dissolved.plot(figsize=(4, 4), column="Route")

#### Overlay
* Need to dissolve at some point??

In [ ]:
# Make sure original columns are retained.
highways_dissolved['original_geometry_hwy'] = highways_dissolved.geometry_buffered

In [ ]:
rail_ca_routes_sjoin['original_geometry_routes'] = rail_ca_routes_sjoin.geometry_buffered

In [ ]:
highways_dissolved = highways_dissolved.to_crs(rail_ca_routes_sjoin.crs)

In [ ]:
# Overlay highways with CA routes
overlay1 = gpd.overlay(
        highways_dissolved, rail_ca_routes_sjoin, how="intersection", keep_geom_type=False
    )


In [ ]:
overlay1.geometry.name

In [ ]:
overlay1.shape

* A route can intersect with multiple highways -> keep only the geometry with the largest area??

In [ ]:
overlay1 = overlay1.assign(
    original_routes_len = overlay1.original_geometry_routes.length,
    new_len = overlay1.geometry.length)

In [ ]:
# Only want geometry of the intersecting route?
# Drop this to make dissolve run faster.
overlay1 = overlay1.drop(columns = ["original_geometry_routes"])

In [ ]:
# Dissolve by district and route, keeping
# only the longest length
overlay_agg = overlay1.dissolve(
     by=["District", "route_id", "route_name"],
     aggfunc={
         "new_len": "max",
         "original_routes_len": "max",
     },
 ).reset_index()

In [ ]:
type(overlay_agg), overlay_agg.route_id.nunique()

#### Categorize
rail routes that are never within 0.25 miles of the SHN

rail routes with > 0 but less than half of its length near the SHN

rail routes with at least half of its length near the SHN

In [ ]:
# Calculate percentage of new length against original one
overlay_agg = overlay_agg.assign(
    percentage = (overlay_agg.new_len / overlay_agg.original_routes_len) * 100)

In [ ]:
def categorize_routes_shn(row):
    if 0 < row.percentage < 50:
        return "less than half of its length near the SHN"
    if row.percentage > 50:
        return "at least half of its length near the SHN"
    else:
        return "never within 0.25 miles of the SHN"

In [ ]:
overlay_agg["shn_category"] = overlay_agg.apply(categorize_routes_shn, axis=1)

In [ ]:
overlay_agg.shn_category.value_counts()

In [ ]:
overlay_agg.groupby(['District','shn_category',]).agg({'route_id':'nunique'})

In [ ]:
overlay_agg.drop(columns = ['geometry'])

In [ ]:
rail_ca_routes_sjoin.columns

In [ ]:
# Plot the 2 original maps together
base = highways_dissolved.plot(figsize=(24, 12),color='white', edgecolor='grey')
rail_ca_routes_sjoin.plot(ax=base, column = 'route_id')

<b>Use a Markdown cell at the end to connect which geospatial concept was applied to which step of the process. The concepts that should be used are projecting CRS, buffering, dissolve, clipping, spatial join, overlay.</b>

* Project CRS: The datasets `routes`,`stops`, and `highways` used to answer this question are originally in the CRS 4326, which is in degrees. The question asks for the intersection between rail routes and highways, which requires a planar projection. As such, I had to project them to EPSG 2229, which is in feet. I also projected CRS when doing combining various GDFS together, as both GDFS need to be in the same CRS.
* Dissolve: 
    * I used `dissolve` to aggregate the `districts` up to the state level. `highways` is also quite granular, as it details which direction the highway goes and I dissolved it as well. 
    * After using `overlay` to find the length of the route that intersects with the highway, I used `dissolve` again. I only wanted the longest length for each District-Route ID-Route Name combination. (I guess I could just sort the gdf by longest length..)
* Clip: I clipped `stops` against `california` to get only the stops in California. 
* Buffer: The goal is to see which routes fall within 1/4 of a mile of a highway. I used `buffer` to buffer the aforementioned amount around each geometry for `highways` and `routes`. 
* Spatial Join: To find out which routes fall in California, I did a `sjoin` between `stops` and `routes`, using an `inner` join and `intersects`. I retained geometry from `routes` instead of `stops` to answer the question.
* Overlay: With the California only routes, I used `overlay` between `routes` and `highways`. I opted for `overlay` because I wanted an altered geometry to calculate the length of the route that actually intersects through the highway. I decided to use `keep_geom_type=False` to return all geometries. 